In [1]:
# Импорты
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow
import json
import gc
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score
)
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn import tree
from sklearn.base import clone

In [2]:
# отключаем экспоненциальное отображение чисел в pandas и numpy и делаем удобное форматирование
def smart_float(x):
    if pd.isnull(x):
        return ""
    elif float(x).is_integer():
        return '{:.0f}'.format(x) # отображаем целые числа без нулевой десятичной части
    else:
        return '{:.6f}'.format(x).rstrip('0').rstrip('.') # отображаем числа с плавающей запятой без лишних нулей

pd.set_option('display.float_format', smart_float)
np.set_printoptions(suppress=True)
# Снимаем ограничение на число отображаемых столбцов в pandas
pd.set_option('display.max_columns', None)      # показывать все столбцы
pd.set_option('display.width', None)            # не ограничивать ширину вывода
pd.set_option('display.max_colwidth', None)     # не ограничивать ширину столбца

In [3]:
# Загружаем результаты из прошлого этапа, будем добавлять новые для сравнения
with open('results2.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

In [3]:
# Функция для добавления результатов модели в общий список и отображения итоговой таблицы
def add_result_prob(name, y_true, y_prob, threshold=0.5):

    y_pred = (y_prob >= threshold).astype(int)

    results.append({
        "Model": name,
        "PR-AUC": average_precision_score(y_true, y_prob),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0)
    })
    
def show_results():

    df = pd.DataFrame(results)

    df = df.sort_values("PR-AUC", ascending=False)

    display(df.round(4))

In [4]:
# Функция для оценки модели на разных порогах и сохранения лучших результатов
def evaluate_model_thresholds(
    model=None, 
    X_validate=None, 
    y_validate=None, 
    X_test=None, 
    y_test=None
):
    if model is None:
        model = globals().get('model')
    if X_validate is None:
        X_validate = globals().get('X_validate')
    if y_validate is None:
        y_validate = globals().get('y_validate')
    if X_test is None:
        X_test = globals().get('X_test')
    if y_test is None:
        y_test = globals().get('y_test')

    val_proba = model.predict_proba(X_validate)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]

    val_pr_auc = average_precision_score(y_validate, val_proba)
    test_pr_auc = average_precision_score(y_test, test_proba)

    val_roc_auc = roc_auc_score(y_validate, val_proba)
    test_roc_auc = roc_auc_score(y_test, test_proba)

    print("=== Метрики ===")
    print(f"VALIDATE PR-AUC: {val_pr_auc:.6f}")
    print(f"TEST     PR-AUC: {test_pr_auc:.6f}")
    print(f"VALIDATE ROC-AUC: {val_roc_auc:.6f}")
    print(f"TEST     ROC-AUC: {test_roc_auc:.6f}")

    thresholds = np.linspace(0.01, 0.99, 100)
    rows = []

    for t in thresholds:
        val_pred = (val_proba >= t).astype(int)
        precision = precision_score(y_validate, val_pred, zero_division=0)
        recall = recall_score(y_validate, val_pred, zero_division=0)
        f1 = f1_score(y_validate, val_pred, zero_division=0)
        rows.append({
            'threshold': t,
            'precision': precision,
            'recall': recall,
            'f1': f1
        })

    threshold_df = pd.DataFrame(rows)

    best_f1_row = threshold_df.loc[threshold_df['f1'].idxmax()]
    best_precision_row = threshold_df.loc[threshold_df['precision'].idxmax()]
    best_recall_row = threshold_df.loc[threshold_df['recall'].idxmax()]

    print("\n=== ЛУЧШИЕ ПОРОГИ (VALIDATE) ===")
    print("\nЛучший F1:")
    print(best_f1_row)
    print("\nЛучший Precision:")
    print(best_precision_row)
    print("\nЛучший Recall:")
    print(best_recall_row)

    best_threshold = best_f1_row['threshold']
    test_pred = (test_proba >= best_threshold).astype(int)
    test_precision = precision_score(y_test, test_pred, zero_division=0)
    test_recall = recall_score(y_test, test_pred, zero_division=0)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)

    print("\n=== ТОП-10 ПОРОГОВ ПО F1 (VALIDATE) ===")
    display(threshold_df.sort_values('f1', ascending=False).head(10))

    print("\n=== ФИНАЛЬНЫЕ МЕТРИКИ TEST (лучший порог по F1) ===")
    print(f"Порог: {best_threshold:.4f}")
    print(f"Precision: {test_precision:.6f}")
    print(f"Recall:    {test_recall:.6f}")
    print(f"F1:        {test_f1:.6f}")

    return best_threshold

## 6.1. Загрузка и подготовка данных

In [5]:
# Загружаем данные для линейной модели
data_linear = pd.read_parquet('data_linear.parquet')

with open('data_linear_schema.json', 'r', encoding='utf-8') as f:
    schema = json.load(f)

# сначала datetime
for col in schema['datetime_cols']:
    if col in data_linear.columns:
        data_linear[col] = pd.to_datetime(data_linear[col], errors='coerce')

# потом category
for col in schema['category_cols']:
    if col in data_linear.columns:
        data_linear[col] = data_linear[col].astype('category')

# потом остальные типы
for col, dtype_str in schema['dtypes'].items():
    if col not in data_linear.columns:
        continue
    
    if col in schema['datetime_cols'] or col in schema['category_cols']:
        continue
    
    try:
        if dtype_str == 'object':
            data_linear[col] = data_linear[col].astype('string')
        else:
            data_linear[col] = data_linear[col].astype(dtype_str)
    except Exception as e:
        print(f'Не удалось привести {col} к {dtype_str}: {e}')
        


In [6]:
# Загружаем данные для деревьев
data_tree = pd.read_parquet('data_tree.parquet')
with open('data_tree_schema.json', 'r', encoding='utf-8') as f:
    schema = json.load(f)

# сначала datetime
for col in schema['datetime_cols']:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors='coerce')

# потом category
for col in schema['category_cols']:
    if col in data_tree.columns:
        data_tree[col] = data_tree[col].astype('category')

# потом остальные типы
for col, dtype_str in schema['dtypes'].items():
    if col not in data_tree.columns:
        continue
    
    if col in schema['datetime_cols'] or col in schema['category_cols']:
        continue
    
    try:
        if dtype_str == 'object':
            data_tree[col] = data_tree[col].astype('string')
        else:
            data_tree[col] = data_tree[col].astype(dtype_str)
    except Exception as e:
        print(f'Не удалось привести {col} к {dtype_str}: {e}')
        
# Делаем MCC категориальным признаком
data_tree['MCC'] = data_tree['MCC'].astype('category')


In [7]:
# Разбиваем данные на train/validate/test для обеих моделей и проводим энкдодинг
target = 'Fraud'
cat_cols = [
    'Merchant_State',
    'Card_Brand',
    'Card_Type',
    'MCC'
]
feature_cols = [c for c in data_linear.columns if c != target]
num_cols = [c for c in feature_cols if c not in cat_cols]
# Линейная
train_linear, validate_linear = train_test_split(
    data_linear, test_size=0.30, shuffle=False
)
validate_linear, test_linear = train_test_split(
    validate_linear, test_size=0.50, shuffle=False
)

X_train_linear = train_linear.drop(columns=[target])
y_train_linear = train_linear[target].astype(int)

X_validate_linear = validate_linear.drop(columns=[target])
y_validate_linear = validate_linear[target].astype(int)

X_test_linear = test_linear.drop(columns=[target])
y_test_linear = test_linear[target].astype(int)

# Создаём ColumnTransformer для числовых и категориальных признаков
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ],
    remainder='drop'
)

# Применяем трансформацию к train, validate и test
X_train_linear = preprocessor.fit_transform(X_train_linear)
X_validate_linear = preprocessor.transform(X_validate_linear)
X_test_linear = preprocessor.transform(X_test_linear)

# Древесная
train_tree, validate_tree = train_test_split(
    data_tree, test_size=0.30, shuffle=False
)
validate_tree, test_tree = train_test_split(
    validate_tree, test_size=0.50, shuffle=False
)

X_train_tree = train_tree.drop(columns=[target])
y_train_tree = train_tree[target].astype(int)

X_validate_tree = validate_tree.drop(columns=[target])
y_validate_tree = validate_tree[target].astype(int)

X_test_tree = test_tree.drop(columns=[target])
y_test_tree = test_tree[target].astype(int)

# Кодируем категориальные признаки для моделей на деревьях
cat_cols = X_train_tree.select_dtypes(include=['category', 'object']).columns.tolist()

ord_encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
X_train_tree[cat_cols] = ord_encoder.fit_transform(X_train_tree[cat_cols])
X_validate_tree[cat_cols] = ord_encoder.transform(X_validate_tree[cat_cols])
X_test_tree[cat_cols] = ord_encoder.transform(X_test_tree[cat_cols])

# Проверка, что длина данных совпадает
assert len(y_validate_linear) == len(y_validate_tree)
assert len(y_test_linear) == len(y_test_tree)
assert y_validate_linear.reset_index(drop=True).equals(y_validate_tree.reset_index(drop=True))
assert y_test_linear.reset_index(drop=True).equals(y_test_tree.reset_index(drop=True))

print("Train linear:", X_train_linear.shape)
print("Validate linear:", X_validate_linear.shape)
print("Test linear:", X_test_linear.shape)

print("Train tree:", X_train_tree.shape)
print("Validate tree:", X_validate_tree.shape)
print("Test tree:", X_test_tree.shape)

Train linear: (5523446, 340)
Validate linear: (1183596, 340)
Test linear: (1183596, 340)
Train tree: (5523446, 54)
Validate tree: (1183596, 54)
Test tree: (1183596, 54)


## 6.2 Обучение мета-модели

Мы не сможем использовать готовое решение в виде StackingClassifier из sklearn т.к. его требование - одинаковые train-наборы для каждой из базовых моделей. Мы не можем это реализовать, поскольку для этого нам придётся либо подавать на "древесную" модель огромное число onehot-признаков либо кодировать категориальные признаки через ordinalencoding или binaryendcoding что не верно методологически (категории в нашей модели не сравнимы в понятии больше-меньше, а бинарное кодирование может быть странные и не стабильные результаты). Кроме того, sklearn не умеет делать стекинг на данных, которые нельзя перемешивать (временные ряды).
Построем собственную модель с OOF-обучением и применением временных рядом.

In [8]:
# Загружаем модели
with open('final_model_log_reg.pkl', 'rb') as f:
    logreg_model = pickle.load(f)
with open('final_rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

print(type(logreg_model))
print(type(rf_model))

<class 'sklearn.linear_model._logistic.LogisticRegression'>
<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [10]:
# predict_proba на validate
val_logreg_proba = logreg_model.predict_proba(X_validate_linear)[:, 1]
val_rf_proba = rf_model.predict_proba(X_validate_tree)[:, 1]

# predict_proba на test
test_logreg_proba = logreg_model.predict_proba(X_test_linear)[:, 1]
test_rf_proba = rf_model.predict_proba(X_test_tree)[:, 1]

# Создаём мета-признаки для модели-стекера
meta_X_validate = pd.DataFrame({
    'logreg_proba': val_logreg_proba,
    'rf_proba': val_rf_proba,
    'proba_diff': val_logreg_proba - val_rf_proba,
    'proba_mean': (val_logreg_proba + val_rf_proba) / 2,
    'proba_max': np.maximum(val_logreg_proba, val_rf_proba),
    'proba_min': np.minimum(val_logreg_proba, val_rf_proba),
})

meta_X_test = pd.DataFrame({
    'logreg_proba': test_logreg_proba,
    'rf_proba': test_rf_proba,
    'proba_diff': test_logreg_proba - test_rf_proba,
    'proba_mean': (test_logreg_proba + test_rf_proba) / 2,
    'proba_max': np.maximum(test_logreg_proba, test_rf_proba),
    'proba_min': np.minimum(test_logreg_proba, test_rf_proba),
})

meta_y_validate = y_validate_linear.reset_index(drop=True)
meta_y_test = y_test_linear.reset_index(drop=True)

display(meta_X_validate.head())

,logreg_proba,rf_proba,proba_diff,proba_mean,proba_max,proba_min
0,0.000002,0,0.000002,0.000001,0.000002,0
1,0.000216,0,0.000216,0.000108,0.000216,0
2,0.000007,0,0.000007,0.000003,0.000007,0
3,0.001565,0,0.001565,0.000782,0.001565,0
4,0.000005,0,0.000005,0.000002,0.000005,0


In [11]:
# Обучаем мета-модель (логистическая регрессия)
meta_model = LogisticRegression(
    solver='lbfgs',
    penalty='l2',
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

meta_model.fit(meta_X_validate, meta_y_validate)

print("Коэффициенты мета-модели:")
display(pd.DataFrame({
    'feature': meta_X_validate.columns,
    'coef': meta_model.coef_[0]
}).sort_values('coef', key=np.abs, ascending=False))

Коэффициенты мета-модели:


,feature,coef
4,proba_max,10.281742
1,rf_proba,7.819129
3,proba_mean,7.383649
0,logreg_proba,6.94817
5,proba_min,4.485556
2,proba_diff,-0.870958


In [12]:
# Подбор порога для мета-модели
best_threshold = evaluate_model_thresholds(
    model=meta_model, 
    X_validate=meta_X_validate, 
    y_validate=meta_y_validate, 
    X_test=meta_X_test, 
    y_test=meta_y_test
)


=== Метрики ===
VALIDATE PR-AUC: 0.816423
TEST     PR-AUC: 0.772079
VALIDATE ROC-AUC: 0.999358
TEST     ROC-AUC: 0.999294

=== ЛУЧШИЕ ПОРОГИ (VALIDATE) ===

Лучший F1:
threshold       0.99
precision   0.441248
recall      0.940401
f1          0.600659
Name: 99, dtype: float64

Лучший Precision:
threshold       0.99
precision   0.441248
recall      0.940401
f1          0.600659
Name: 99, dtype: float64

Лучший Recall:
threshold       0.01
precision   0.001474
recall             1
f1          0.002944
Name: 0, dtype: float64

=== ТОП-10 ПОРОГОВ ПО F1 (VALIDATE) ===


,threshold,precision,recall,f1
99,0.99,0.441248,0.940401,0.600659
98,0.980101,0.404396,0.948997,0.567123
97,0.970202,0.37738,0.954155,0.540848
96,0.960303,0.350807,0.958739,0.513663
95,0.950404,0.32934,0.961032,0.490566
94,0.940505,0.30837,0.962751,0.467121
93,0.930606,0.292035,0.96447,0.448322
92,0.920707,0.279139,0.966189,0.433141
91,0.910808,0.26818,0.967908,0.419993
90,0.900909,0.258568,0.968481,0.408163



=== ФИНАЛЬНЫЕ МЕТРИКИ TEST (лучший порог по F1) ===
Порог: 0.9900
Precision: 0.397706
Recall:    0.930129
F1:        0.557174


In [13]:
# Результаты получились плохими - очень завышенный recall и низкий precision, что говорит о том, что мета-модель слишком аггресивна
# Проведем небольшую диагностику
diag = pd.DataFrame({
    'y': y_validate_linear.reset_index(drop=True),
    'logreg_proba': val_logreg_proba,
    'rf_proba': val_rf_proba
})

print("Correlation between base model probabilities:")
print(diag[['logreg_proba', 'rf_proba']].corr())

print("\nValidate PR-AUC:")
print("LogReg:", average_precision_score(diag['y'], diag['logreg_proba']))
print("RF    :", average_precision_score(diag['y'], diag['rf_proba']))

print("\nValidate ROC-AUC:")
print("LogReg:", roc_auc_score(diag['y'], diag['logreg_proba']))
print("RF    :", roc_auc_score(diag['y'], diag['rf_proba']))

Correlation between base model probabilities:
              logreg_proba  rf_proba
logreg_proba             1  0.536582
rf_proba          0.536582         1

Validate PR-AUC:
LogReg: 0.38475604251498585
RF    : 0.8606082377812252

Validate ROC-AUC:
LogReg: 0.9861599224812709
RF    : 0.9993858785921406


Как видим, модели слишком разные - логистическая регрессия (базовая) вносит слишком сильный шум в мета-модель. Вместе с тем правильная компбинация была бы очень полезна, ведь, как мы выяснили ранее, у каждой модели есть свои приемущества. Попробуем улучшить обучение мета-модели.

In [16]:
# Попробуем сделать OOF обучение каждой из моделей и последующее обучение мета-модели на этих данных
# Получаем гиперпараметры моделей
params_logreg = logreg_model.get_params()
params_rf = rf_model.get_params()
logreg_base = LogisticRegression(**params_logreg)
rf_base = RandomForestClassifier(**params_rf)

# Создаём OOF-предсказания для каждой модели
tscv = TimeSeriesSplit(n_splits=3)

oof_logreg = np.full(X_train_linear.shape[0], np.nan)
oof_rf = np.full(X_train_tree.shape[0], np.nan)

for fold, (tr_idx, val_idx) in enumerate(tscv.split(np.arange(X_train_linear.shape[0])), 1):
    print(f'Fold {fold}')

    # linear model
    model_logreg = clone(logreg_base)
    model_logreg.fit(
        X_train_linear[tr_idx],
        y_train_linear.iloc[tr_idx]
    )
    oof_logreg[val_idx] = model_logreg.predict_proba(
        X_train_linear[val_idx]
    )[:, 1]

    # tree model
    model_rf = clone(rf_base)
    model_rf.fit(
        X_train_tree.iloc[tr_idx],
        y_train_tree.iloc[tr_idx]
    )
    oof_rf[val_idx] = model_rf.predict_proba(
        X_train_tree.iloc[val_idx]
    )[:, 1]

# Оставляем только строки, которые реально получили OOF-предсказание
mask = ~np.isnan(oof_logreg) & ~np.isnan(oof_rf)

meta_X_train = pd.DataFrame({
    'logreg_proba': oof_logreg[mask],
    'rf_proba': oof_rf[mask],
    'proba_diff': oof_logreg[mask] - oof_rf[mask],
    'proba_mean': (oof_logreg[mask] + oof_rf[mask]) / 2,
    'proba_max': np.maximum(oof_logreg[mask], oof_rf[mask]),
    'proba_min': np.minimum(oof_logreg[mask], oof_rf[mask]),
    'abs_diff': np.abs(oof_logreg[mask] - oof_rf[mask]),
})

meta_y_train = y_train_linear.iloc[mask].reset_index(drop=True)

print("Meta-train shape:", meta_X_train.shape)
print("OOF LogReg PR-AUC:", average_precision_score(meta_y_train, meta_X_train['logreg_proba']))
print("OOF RF PR-AUC:", average_precision_score(meta_y_train, meta_X_train['rf_proba']))

Fold 1
Fold 2
Fold 3
Meta-train shape: (4142583, 7)
OOF LogReg PR-AUC: 0.48404403803647617
OOF RF PR-AUC: 0.5587297947977974


In [17]:
# Обучаем мета-модель на OOF-предсказаниях
meta_model = LogisticRegression(
    solver='lbfgs',
    penalty='l2',
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

meta_model.fit(meta_X_train, meta_y_train)

coef_df = pd.DataFrame({
    'feature': meta_X_train.columns,
    'coef': meta_model.coef_[0]
}).sort_values('coef', key=np.abs, ascending=False)

display(coef_df)

,feature,coef
4,proba_max,6.649503
6,abs_diff,6.109765
1,rf_proba,5.925327
2,proba_diff,-4.661413
3,proba_mean,3.59462
0,logreg_proba,1.263914
5,proba_min,0.539738


In [18]:
#  Переобучаем базовые модели на всем train
final_logreg = clone(logreg_base)
final_rf = clone(rf_base)

final_logreg.fit(X_train_linear, y_train_linear)
final_rf.fit(X_train_tree, y_train_tree)

test_logreg_proba = final_logreg.predict_proba(X_test_linear)[:, 1]
test_rf_proba = final_rf.predict_proba(X_test_tree)[:, 1]

meta_X_test = pd.DataFrame({
    'logreg_proba': test_logreg_proba,
    'rf_proba': test_rf_proba,
    'proba_diff': test_logreg_proba - test_rf_proba,
    'proba_mean': (test_logreg_proba + test_rf_proba) / 2,
    'proba_max': np.maximum(test_logreg_proba, test_rf_proba),
    'proba_min': np.minimum(test_logreg_proba, test_rf_proba),
    'abs_diff': np.abs(test_logreg_proba - test_rf_proba),
})

stack_test_proba = meta_model.predict_proba(meta_X_test)[:, 1]

print("Stacking TEST PR-AUC:", average_precision_score(y_test_linear, stack_test_proba))
print("Stacking TEST ROC-AUC:", roc_auc_score(y_test_linear, stack_test_proba))

Stacking TEST PR-AUC: 0.7199497016613359
Stacking TEST ROC-AUC: 0.999215770195321


In [19]:
# Подбор порога для мета-модели на OOF-данных
thresholds = np.linspace(0.01, 0.99, 99)
meta_train_proba = meta_model.predict_proba(meta_X_train)[:, 1]

rows = []
for t in thresholds:
    pred = (meta_train_proba >= t).astype(int)
    rows.append({
        'threshold': t,
        'precision': precision_score(meta_y_train, pred, zero_division=0),
        'recall': recall_score(meta_y_train, pred, zero_division=0),
        'f1': f1_score(meta_y_train, pred, zero_division=0),
    })

thr_df = pd.DataFrame(rows).sort_values('f1', ascending=False)
display(thr_df.head(10))

best_threshold = thr_df.iloc[0]['threshold']
print("Best threshold:", best_threshold)

,threshold,precision,recall,f1
98,0.99,0.11971,0.694866,0.204235
97,0.98,0.112998,0.702278,0.194673
96,0.97,0.108625,0.708062,0.188354
95,0.96,0.105393,0.711135,0.183579
94,0.95,0.102647,0.714389,0.179502
93,0.94,0.100417,0.717462,0.176177
92,0.93,0.098363,0.71927,0.17306
91,0.92,0.09651,0.7218,0.170255
90,0.91,0.094927,0.72415,0.16785
89,0.9,0.093279,0.725597,0.165308


Best threshold: 0.99


In [20]:
stack_test_pred = (stack_test_proba >= best_threshold).astype(int)

print("Accuracy :", accuracy_score(y_test_linear, stack_test_pred))
print("Precision:", precision_score(y_test_linear, stack_test_pred, zero_division=0))
print("Recall   :", recall_score(y_test_linear, stack_test_pred, zero_division=0))
print("F1       :", f1_score(y_test_linear, stack_test_pred, zero_division=0))
print(confusion_matrix(y_test_linear, stack_test_pred))

Accuracy : 0.9928455317523884
Precision: 0.17134140340517665
Recall   : 0.9731693683622136
F1       : 0.2913807531380753
[[1173387    8420]
 [     48    1741]]


In [21]:
add_result_prob("Стекинг (LogReg + RF, OOF)", y_test_linear, stack_test_proba, threshold=best_threshold)
show_results()

,Model,PR-AUC,ROC-AUC,Accuracy,Precision,Recall,F1
20,"Random Forest (Ручная оптимизация, тест)",0.7865,0.9993,0.9992,0.7582,0.7149,0.7359
19,"Random Forest (после Optuna, тест)",0.7791,0.9993,0.9991,0.814,0.4891,0.611
18,"Случайный лес (с подобраными гипер-параметрами, test)",0.7609,0.9992,0.9991,0.7221,0.6551,0.687
22,"Стекинг (LogReg + RF, OOF)",0.7199,0.9992,0.9928,0.1713,0.9732,0.2914
17,Базовый RF 500 деревьев (Test),0.718,0.9992,0.999,0.6786,0.6138,0.6446
21,"Random Forest (признаки отобраны через logreg, тест)",0.6388,0.9982,0.9988,0.6044,0.6003,0.6024
9,Логистическая регрессия (без подбора гиперпараметров) на тесте,0.5957,0.9963,0.9989,0.7781,0.398,0.5266
11,"Логистическая регрессия (без подбора гиперпараметров) обучена на train+validate, тестирование на test",0.5957,0.9963,0.9989,0.7781,0.398,0.5266
10,Логистическая регрессия (после подбора гиперпараметров) на тесте,0.5813,0.9958,0.9988,0.6226,0.4925,0.5499
12,"Логистическая регрессия (после подбора гиперпараметров) обучена на train+validate, тестирование на test",0.5813,0.9958,0.9987,0.5781,0.5254,0.5505


In [22]:
# Сохраним список результатов
with open('results2.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

Как видим, стекинг не показзал хорошего результата. Фактически, его текущий результат находится на уровне базового случайного леса на 500 деревьев без подбора гипер-параметров. Учитывая огромный порог можем сделать вывод, что стакинг, фактически, полагается только на древесную модель, почти полностью игнорируя вклад логистической регресии. 
На каждые 100,000 транзакций модель выдаст около 809 алертов, при этом 670 будут ложными, а только около 139 — настоящими. Зато пропущенных будет очень мало. Однако, такая модель не будет операционно эффективное в реальном банке - если алёрты будут проверятся в ручную - это большие затраты на персонал, если блокировки будут автоматические - это огромнео число недовольных клиентов, которым заблокировало карточку когда они попытались за границей в "опасной" стране оплатить картой покупку хот-дога. 